# Task 4.3: effect of PARRM on ridge refit

Because unsupervised PARRM artifact removal fails to restore the accuracy of an OFF-trained ridge decoder (Task 3), this notebook tests whether PARRM improves decoding accuracy when refitting on $k$ labeled ON trials. Each session supplies a 20-trial calibration pool and a fixed 30-trial query (Section 4.1), so we refit separate ridge decoders on raw and PARRM-cleaned features with the same draws and the same penalty, which isolates the effect of artifact removal.

## PARRM

During Butters' reaching task, motor cortex local field potentials (0.1–300 Hz) recorded by a 64-electrode array are dominated by a periodic stimulation artifact from a partially demodulated 3.5 kHz carrier modulated at 3 Hz or 20 Hz. 

PARRM removes this artifact without labels by phase-averaging across cycles to subtract a waveform template, which neither restores OFF-trained decoders (Task 3.2) nor improves decoders refitted on k labeled ON trials. 

Refitting with PARRM fails to help because within-session artifact drift leaves large residuals (up to z ≈ 5300), template subtraction removes genuine phase-locked neural and movement signals, and ridge regression already down-weights artifact-dominated channels on its own.

## Setup

In [ ]:
import os
import sys

HERE = os.path.abspath(os.getcwd())
TASK4_DIR = HERE if os.path.basename(HERE) == 'task4' else os.path.join(HERE, 'task4')
TASK3_DIR = os.path.join(os.path.dirname(TASK4_DIR), 'task3')
for path in (TASK4_DIR, TASK3_DIR):
    if path not in sys.path:
        sys.path.insert(0, path)

import utils
import task3_common  # ridge solver, penalty grid, clipping and PARRM map shared with task 3

print('task 4 helpers:', utils.TASK4_DIR)
print('task 3 helpers:', task3_common.TASK_DIR)
print('draws file    :', utils.DRAWS_PATH)


task 4 helpers: /Volumes/Kevin's SSD/Classes/CS182/EECS182-Project/task4
task 3 helpers: /Volumes/Kevin's SSD/Classes/CS182/EECS182-Project/task3
draws file    : /Volumes/Kevin's SSD/Classes/CS182/EECS182-Project/task4/results/draws.json


In [ ]:
import numpy as np
import pandas as pd

LAG_COUNT = utils.LAG_COUNT
FEATURE_VARIANTS = ((utils.RAW_VARIANT, 'raw'), (utils.PARRM_VARIANT, 'parrm'))
DRAWS = utils.load_json(utils.DRAWS_PATH)

def clip(windows):
    """Clamp z-scored features to +/-CLIP_Z so one artifact bin cannot dominate the weights."""
    X, y = windows
    return np.clip(X, -task3_common.CLIP_Z, task3_common.CLIP_Z), y

print('draws:', len(DRAWS['sessions']), 'sessions | ks', DRAWS['ks'],
      '| query definition', DRAWS.get('query_definition'))


draws: 14 sessions | ks [2, 5, 10, 20] | query definition remaining


## Protocol

Everything comes from `task3_common`, so task 3 and task 4.3 make identical choices and only
the fit data differs. That covers the ridge solver, the penalty grid
(`task3_common.LAMBDAS`), and the clipping (`task3_common.CLIP_Z`). Features are clipped to
$\pm 10$ so one artifact bin can't dominate the weights. The clip binds almost only on raw
features under stimulation, so if anything it handicaps the raw arm, and PARRM still doesn't
beat it.

$\lambda$ is picked once on the ON validation sessions and held for every draw. Each draw
normalizes on its own trials (see 4.1). Raw and PARRM share the same draws and query trials.
Each choice removes a confound, leaving $k$ and the input variant as the only things that
vary, which is the experiment.

In [ ]:
X_train, y_train = clip(utils.concat_session_windows(utils.OFF_TRAINING_SESSIONS, 'train', utils.RAW_VARIANT, LAG_COUNT))

validation = [clip(utils.windows_from_valid_bins(utils.load_split_dataset(s, 'query', utils.RAW_VARIANT, LAG_COUNT)))
              for s in utils.sessions_in_split('val', utils.STIMULATED_ON)]

fitter = task3_common.RidgeFitter(X_train, y_train)

scores = {float(lam): np.mean([utils.mean_r2(y, fitter.fit(lam).predict(X)) for X, y in validation])
          for lam in task3_common.LAMBDAS}

alpha = max(scores, key=scores.get)
print('ridge alpha:', alpha, '(from task 3 grid of', len(task3_common.LAMBDAS), ')')
pd.Series(scores).round(4)


ridge alpha: 100000.0 (from task 3 grid of 13 )


1.000000         -0.5760
3.162278         -0.5109
10.000000        -0.4159
31.622777        -0.3002
100.000000       -0.1876
316.227766       -0.0959
1000.000000      -0.0297
3162.277660       0.0090
10000.000000      0.0264
31622.776602      0.0315
100000.000000     0.0320
316227.766017     0.0311
1000000.000000    0.0270
dtype: float64

In [ ]:
def refit_draw(session, k, index, draw, variant, feature_label, alpha):
    X_cal, y_cal = clip(utils.drawn_trial_windows(session, draw, variant, LAG_COUNT))
    X_query, y_query = clip(utils.query_trial_windows(session, draw, variant, LAG_COUNT))
    off = clip(utils.windows_from_valid_bins(utils.load_same_day_off_dataset(session, variant, LAG_COUNT)))
    model = task3_common.ridge_path(X_cal, y_cal, [alpha])[0]  # task 3 solver, one lambda
    return utils.score_window_decoder(model, session, 'ridge', feature_label,
                           (X_query, y_query), off, k=k, draw=index)

rows = []
for session in DRAWS['sessions']:
    for k, index, draw in utils.iter_draws(DRAWS, session):
        for variant, feature_label in FEATURE_VARIANTS:
            rows.append(refit_draw(session, k, index, draw, variant, feature_label, alpha))
    print('scored', session)

results = pd.DataFrame(rows)
results.to_csv(os.path.join(utils.TASK4_RESULTS, 'task4_3_parrm_ridge.csv'), index=False)
print(results.shape)
results.groupby(['feature_variant', 'k'])[['r2_ave', 'r2_off_ave']].mean().round(3)


## Recovery per $k$

Each row is one (variant, $k$): the mean over 14 sessions, with a 95% CI that resamples
sessions and, inside each session, its calibration draws. The query trials are fixed for
every draw, so the interval covers session and calibration choice rather than query-set
luck. At $k=20$ the calibration set is the whole 20-trial pool and there is one draw, not
ten. `r2_off` is the OFF control from above.

Read the $k$ axis, not the gap between rows. A PARRM-vs-raw difference of 0.001 is
invisible at this scale, while the rise from $k=2$ to $k=20$ is 30-50x larger.

In [ ]:
summary = []
for (variant, k), part in results.groupby(['feature_variant', 'k']):
    groups = [group.r2_ave.to_numpy() for _, group in part.groupby('session')]
    off_groups = [group.r2_off_ave.to_numpy() for _, group in part.groupby('session')]
    low, high = utils.two_level_bootstrap_confidence_interval(groups)
    summary.append(dict(feature_variant=variant, k=k, sessions=len(groups),
                        draws_per_session=len(groups[0]),
                        r2=float(np.mean([group.mean() for group in groups])),
                        ci_low=low, ci_high=high,
                        r2_off=float(np.mean([group.mean() for group in off_groups]))))
summary = pd.DataFrame(summary)
summary.to_csv(os.path.join(utils.TASK4_RESULTS, 'task4_3_summary.csv'), index=False)
summary.round(4)